# 365 Probabilidades — Dia #034
## Qual a probabilidade de menos exercício ser melhor pro seu humor do que você imagina?

**Tipo:** Comportamental / Fisiologia
**Data de publicação:** 2026-07-17
**Ferramenta:** Python
**Decisão analisada:** Quando o treino não cabe na semana, quanto de benefício mental eu realmente estou perdendo?
**Hashtag:** #365Probabilidades #Dia034

---

### 📖 A História

Faz dez dias que eu não treino.

Não é por preguiça, nem por lesão. É por vida. Nas últimas semanas, as mudanças que estão acontecendo por aqui (algumas eu já contei em outros dias) simplesmente empurraram a corrida pra fora da lista.

Pra quem me acompanha e sabe que correr é parte importante de quem eu sou, essa frase provavelmente já cria uma expectativa. A tentativa de me convencer de que "é só uma fase", que "amanhã eu retomo", que "eu preciso urgente voltar", porque, afinal, todo mundo sabe que exercício faz bem pra cabeça.

Menos gente sabe **quanto** faz bem, e ainda menos sabe **quanto** é suficiente.

Eu abri esses dez dias sem treino com a sensação vaga de estar perdendo algo importante. Um humor melhor. Uma clareza mental. Uma noite de sono mais funda. Coisas que a gente aprende a associar automaticamente com movimento.

Aí, no meio dessa culpa esportiva de baixa intensidade, resolvi fazer o que faço em todos os outros dias deste projeto: procurar os dados. Quanto tempo de exercício, na prática, muda o humor de verdade? E, mais importante, existe um ponto em que **mais** exercício deixa de ajudar?

O que encontrei foi, ao mesmo tempo, um alívio e uma reorganização mental. Não é o post que eu esperava escrever no fim de dez dias sem correr. É melhor.

---

### 📚 O Conceito: A Curva em U

Em 2018, um grupo liderado por Adam Chekroud, da Yale School of Medicine, publicou no Lancet Psychiatry o maior estudo já feito sobre exercício e saúde mental. A amostra: um milhão e duzentos mil adultos americanos, cobrindo três anos de dados do Behavioral Risk Factor Surveillance System (BRFSS 2011, 2013 e 2015).

Não é uma pesquisa pequena. É praticamente uma varredura populacional.

A pergunta central era simples e ambiciosa. Quanto tempo de exercício, com que frequência, e de que tipo, está associado a menos dias de sofrimento mental por mês? A métrica de desfecho não é humor difuso, é concreta: quantos dias, no último mês, você sentiu sua saúde mental "não boa"?

E o achado que virou notícia no mundo inteiro é o que segue: a relação entre exercício e humor não é linear. Ela tem formato de U. Fazer nada é ruim. Fazer pouco é melhor. Fazer na medida certa é ainda melhor. Mas passar dessa medida, o benefício começa a diminuir. E, num certo ponto, exercitar demais fica **pior** do que não se exercitar.

O ponto ótimo, no estudo, é surpreendentemente modesto: **45 minutos por sessão, 3 a 5 vezes por semana**. Não é ultra-atleta. É gente comum.

E a cauda longa do gráfico é o que mais me pegou. Sessões acima de 90 minutos começam a perder benefício mental. Mais de 23 sessões por mês, também. E sessões acima de 3 horas se associam a mais sofrimento mental do que ficar parado.

Isso não invalida a maratona nem o treinamento intenso. São contextos com outros benefícios (físicos, competitivos, identitários). Mas se a métrica é apenas humor, o retorno decrescente da carga é um dado real.

---

### 🧮 O Modelo

**Fonte principal:**
- Chekroud, S. R., Gueorguieva, R., Zheutlin, A. B., Paulus, M., Krumholz, H. M., Krystal, J. H. & Chekroud, A. M. (2018) — "Association between physical exercise and mental health in 1·2 million individuals in the USA between 2011 and 2015: a cross-sectional study", *The Lancet Psychiatry*, 5(9), 739-746 — **N=1.237.194 adultos**, dados BRFSS 2011/2013/2015

**Fontes de reforço:**
- Reed, J. & Ones, D. S. (2006) — "The effect of acute aerobic exercise on positive activated affect: A meta-analysis", *Psychology of Sport and Exercise*, 7(5), 477-514 — meta-análise clássica sobre efeitos agudos
- Chan, J. S. Y., Liu, G., Liang, D., Deng, K., Wu, J. & Yan, J. H. (2019) — meta-análise sobre efeitos agudos de curta duração no humor

**Nota metodológica:** neste dia **não aplico o fator de correção ×0.80**. O achado central não é uma proporção populacional de survey extrapolada, é uma associação estatística de coorte gigantesca (N=1,2 milhão), ajustada por variáveis sociodemográficas e físicas. É a mesma categoria de dado do #032 (leitura e longevidade).

A limitação relevante é outra e ela precisa ficar clara: o estudo é *cross-sectional*, ou seja, mostra associação num ponto no tempo, não relação causal direta. Não conseguimos afirmar categoricamente que "quem malha 3h por sessão fica pior por causa da malhação", mas a associação está lá e é forte. E os dados são de saúde mental autorrelatada, com uma métrica específica (dias de "sofrimento mental" no último mês).


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")


Bibliotecas carregadas


In [2]:
# --- DADOS DA LITERATURA ---
# Chekroud et al. (2018), The Lancet Psychiatry 5(9):739-746
# Cross-sectional, dados BRFSS 2011/2013/2015

n_estudo = 1237194

# Efeito geral de exercitar-se (vs nao exercitar-se)
dias_reducao_geral = 1.49        # 1,49 dias a menos de sofrimento mental por mes
p_reducao_geral = 0.432          # 43,2% de reducao

# Efeito em pessoas com historico de depressao (maior)
dias_reducao_depressao = 3.75
p_reducao_depressao = 0.345

# Reducao por tipo de exercicio (relativo a nao exercitar-se)
reducao_por_tipo = {
    'Esportes em equipe': 0.223,   # 22,3%
    'Ciclismo': 0.216,             # 21,6%
    'Aerobico / academia': 0.201,  # 20,1%
    'Caminhada': 0.177,            # 17,7%
}

# Ponto otimo
duracao_otima_min = 45           # minutos por sessao
freq_otima_min = 3               # vezes por semana
freq_otima_max = 5

# Pontos de degradacao (a partir de onde comeca a piorar)
duracao_degradacao_min = 90      # sessoes acima disso comecam a perder beneficio
duracao_pior_que_nada = 180      # acima de 3h por sessao piora
sessoes_max_por_mes = 23         # mais que isso comeca a piorar

# NOTA: nao aplico fator de correcao x0.80 neste dia.
# E associacao de coorte gigantesca (N=1,2 milhao), nao proporcao de survey.
# Limitacao a declarar: estudo cross-sectional, mostra associacao, nao causa direta.

print("=" * 68)
print("  DADOS — EXERCICIO E SAUDE MENTAL")
print("=" * 68)
print(f"\n  Chekroud et al. (2018), The Lancet Psychiatry")
print(f"  N = {n_estudo:,} adultos (BRFSS 2011/2013/2015)")

print(f"\n  EFEITO GERAL:")
print(f"  Quem se exercita reporta {dias_reducao_geral} dias a menos")
print(f"  de sofrimento mental por mes.")
print(f"  Reducao de {p_reducao_geral*100:.1f}% vs quem nao se exercita.")

print(f"\n  EFEITO EM QUEM TEM HISTORICO DE DEPRESSAO:")
print(f"  {dias_reducao_depressao} dias a menos por mes")
print(f"  Reducao de {p_reducao_depressao*100:.1f}%")

print(f"\n  REDUCAO POR TIPO (vs nao exercitar-se):")
for tipo, reducao in reducao_por_tipo.items():
    print(f"    {tipo:<25}    -{reducao*100:.1f}%")

print(f"\n  PONTO OTIMO:")
print(f"  Duracao:                    {duracao_otima_min} min por sessao")
print(f"  Frequencia:                 {freq_otima_min}-{freq_otima_max} x por semana")

print(f"\n  ATENCAO: mais nem sempre e melhor")
print(f"  > {duracao_degradacao_min} min por sessao: beneficio diminui")
print(f"  > {duracao_pior_que_nada} min (3h) por sessao: PIOR que nao se exercitar")
print(f"  > {sessoes_max_por_mes} sessoes por mes: beneficio diminui")
print("=" * 68)


  DADOS — EXERCICIO E SAUDE MENTAL

  Chekroud et al. (2018), The Lancet Psychiatry
  N = 1,237,194 adultos (BRFSS 2011/2013/2015)

  EFEITO GERAL:
  Quem se exercita reporta 1.49 dias a menos
  de sofrimento mental por mes.
  Reducao de 43.2% vs quem nao se exercita.

  EFEITO EM QUEM TEM HISTORICO DE DEPRESSAO:
  3.75 dias a menos por mes
  Reducao de 34.5%

  REDUCAO POR TIPO (vs nao exercitar-se):
    Esportes em equipe           -22.3%
    Ciclismo                     -21.6%
    Aerobico / academia          -20.1%
    Caminhada                    -17.7%

  PONTO OTIMO:
  Duracao:                    45 min por sessao
  Frequencia:                 3-5 x por semana

  ATENCAO: mais nem sempre e melhor
  > 90 min por sessao: beneficio diminui
  > 180 min (3h) por sessao: PIOR que nao se exercitar
  > 23 sessoes por mes: beneficio diminui


In [3]:
# --- O MODELO ---
# A curva em U e a leitura pra vida real

# Simulacao ilustrativa da curva em U (dias de sofrimento mental por mes)
# como funcao da duracao de sessao
duracao_min = np.array([0, 15, 30, 45, 60, 75, 90, 105, 120, 150, 180, 240])
# Dias de sofrimento mental (valores ilustrativos consistentes com o padrao)
dias_curva = np.array([3.9, 3.2, 2.7, 2.4, 2.5, 2.8, 3.2, 3.6, 4.0, 4.5, 4.8, 5.0])

print("=" * 68)
print("  MODELO — A CURVA EM U E O QUE ISSO SIGNIFICA")
print("=" * 68)
print(f"\n  A relacao entre duracao de exercicio e saude mental")
print(f"  NAO e linear. Tem formato de U:")
print(f"\n  {'Duracao (min)':<15} {'~Dias sofrim. mental/mes':>28}")
for d, s in zip(duracao_min, dias_curva):
    marker = "  <- OTIMO" if d == 45 else ("  <- PIOR QUE NADA" if d >= 180 else "")
    print(f"  {d:<15} {s:>15.1f}{marker}")

print(f"\n  LEITURA PRATICA:")
print(f"  Zero minutos: humor pior (fazer nada nao e otimo)")
print(f"  45 minutos:   melhor humor (ponto otimo)")
print(f"  90 minutos:   ja perdeu boa parte do beneficio")
print(f"  180 minutos:  pior que ficar parado")

print(f"\n  O RECADO DIFICIL:")
print(f"  A regua do 'mais e sempre melhor' esta errada.")
print(f"  Exercicio e adaptacao. Adaptacao exige recuperacao.")
print(f"  Excesso vira o oposto do proposito.")

print(f"\n  PRA QUEM ESTA FORA DA JANELA OTIMA:")
print(f"  Ate CAMINHAR reduz 17,7% dos dias de sofrimento mental.")
print(f"  A versao pequena de voltar ao movimento ja captura")
print(f"  a maior parte do beneficio descrito.")
print("=" * 68)


  MODELO — A CURVA EM U E O QUE ISSO SIGNIFICA

  A relacao entre duracao de exercicio e saude mental
  NAO e linear. Tem formato de U:

  Duracao (min)       ~Dias sofrim. mental/mes
  0                           3.9
  15                          3.2
  30                          2.7
  45                          2.4  <- OTIMO
  60                          2.5
  75                          2.8
  90                          3.2
  105                         3.6
  120                         4.0
  150                         4.5
  180                         4.8  <- PIOR QUE NADA
  240                         5.0  <- PIOR QUE NADA

  LEITURA PRATICA:
  Zero minutos: humor pior (fazer nada nao e otimo)
  45 minutos:   melhor humor (ponto otimo)
  90 minutos:   ja perdeu boa parte do beneficio
  180 minutos:  pior que ficar parado

  O RECADO DIFICIL:
  A regua do 'mais e sempre melhor' esta errada.
  Exercicio e adaptacao. Adaptacao exige recuperacao.
  Excesso vira o oposto do proposito

In [4]:
# --- VISUALIZACAO ---

cor_otimo = '#2a8a82'
cor_alerta = '#c0392b'
cor_ouro = '#c8a84b'
cor_neutra = '#9c9b94'

# GRAFICO 1 — A curva em U (o achado central)
fig1, ax1 = plt.subplots(figsize=(11, 6.5))

# Suavizar com interpolacao
from scipy.interpolate import make_interp_spline
duracao_smooth = np.linspace(duracao_min.min(), duracao_min.max(), 300)
spl = make_interp_spline(duracao_min, dias_curva, k=3)
dias_smooth = spl(duracao_smooth)

ax1.plot(duracao_smooth, dias_smooth, color=cor_alerta, linewidth=3)
ax1.fill_between(duracao_smooth, dias_smooth, alpha=0.12, color=cor_alerta)

# Marcar ponto otimo
idx_45 = np.argmin(np.abs(duracao_min - 45))
ax1.plot(45, dias_curva[idx_45], 'o', color=cor_otimo, markersize=14, zorder=5)
ax1.annotate('OTIMO\n45 min',
             xy=(45, dias_curva[idx_45]), xytext=(45, 1.4),
             fontsize=12, color=cor_otimo, fontweight='bold', ha='center',
             arrowprops=dict(arrowstyle='->', color=cor_otimo, lw=1.5))

# Marcar ponto zero (fazer nada)
ax1.plot(0, dias_curva[0], 's', color=cor_neutra, markersize=11, zorder=5)
ax1.annotate('fazer nada',
             xy=(0, dias_curva[0]), xytext=(10, 4.4),
             fontsize=10, color='#555', style='italic', ha='left',
             arrowprops=dict(arrowstyle='->', color='#888', lw=1))

# Marcar ponto de 180 min
idx_180 = np.argmin(np.abs(duracao_min - 180))
ax1.plot(180, dias_curva[idx_180], 'X', color=cor_alerta, markersize=13, zorder=5)
ax1.annotate('3h por sessao:\nPIOR que nada',
             xy=(180, dias_curva[idx_180]), xytext=(210, 4.4),
             fontsize=11, color=cor_alerta, fontweight='bold', ha='left',
             arrowprops=dict(arrowstyle='->', color=cor_alerta, lw=1.2))

# Sombrear zona otima (30-60 min)
ax1.axvspan(30, 60, alpha=0.10, color=cor_otimo)

ax1.set_xlabel('Duracao da sessao (minutos)')
ax1.set_ylabel('Dias de "sofrimento mental" no ultimo mes')
ax1.set_ylim(1, 5.5)
ax1.set_xlim(-10, 250)
ax1.set_title('A curva em U do exercicio\nMais nao e sempre melhor. Existe um ponto otimo, e ele e modesto.',
              fontsize=13, pad=15)

plt.figtext(0.5, 0.005,
            'Curva ilustrativa consistente com os achados de Chekroud et al. (2018), Lancet Psychiatry, N=1,2 milhao | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-034-grafico-01-curva-u.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo!")

# GRAFICO 2 — Reducao por tipo de exercicio (barras horizontais)
fig2, ax2 = plt.subplots(figsize=(11, 6.5))

tipos = list(reducao_por_tipo.keys())
reducoes = [reducao_por_tipo[t]*100 for t in tipos]

# Ordenar do maior pro menor
idx_sort = np.argsort(reducoes)
tipos_sorted = [tipos[i] for i in idx_sort]
reducoes_sorted = [reducoes[i] for i in idx_sort]

y_pos = np.arange(len(tipos_sorted))
cores = [cor_neutra if r < 20 else cor_otimo for r in reducoes_sorted]

bars = ax2.barh(y_pos, reducoes_sorted, color=cores, alpha=0.88, height=0.6)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(tipos_sorted, fontsize=12)
ax2.set_xlabel('Reducao nos dias de sofrimento mental (%)')
ax2.set_xlim(0, 27)
ax2.set_title('Todo movimento conta\nAte caminhar reduz 17,7% dos dias de sofrimento mental',
              fontsize=13, pad=15)

for bar, v in zip(bars, reducoes_sorted):
    ax2.text(v + 0.4, bar.get_y() + bar.get_height()/2,
             f'-{v:.1f}%', va='center', fontweight='bold', fontsize=13)

plt.figtext(0.5, 0.005,
            'Fonte: Chekroud et al. (2018), Lancet Psychiatry — reducao vs quem nao se exercita | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-034-grafico-02-por-tipo.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo!")

# GRAFICO 3 — Efeito adicional em quem tem historico de depressao
fig3, ax3 = plt.subplots(figsize=(11, 6.5))

categorias = ['Populacao geral', 'Com historico\nde depressao']
dias_reduzidos = [dias_reducao_geral, dias_reducao_depressao]
cores3 = [cor_neutra, cor_otimo]

bars3 = ax3.bar(categorias, dias_reduzidos, color=cores3, alpha=0.88, width=0.45)
ax3.set_ylabel('Dias a menos de sofrimento mental por mes')
ax3.set_ylim(0, 5)
ax3.set_title('O beneficio e ainda maior em quem tem historico de depressao\nEm media, 3,75 dias a menos de sofrimento mental por mes',
              fontsize=13, pad=15)

for bar, v in zip(bars3, dias_reduzidos):
    ax3.text(bar.get_x() + bar.get_width()/2, v + 0.1,
             f'-{v} dias', ha='center', fontweight='bold', fontsize=16)

plt.figtext(0.5, 0.005,
            'Fonte: Chekroud et al. (2018), Lancet Psychiatry — vs quem nao se exercita | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-034-grafico-03-depressao.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo!")


Grafico 1 salvo!
Grafico 2 salvo!
Grafico 3 salvo!


### 💡 O Insight

**O ponto ótimo do exercício pra saúde mental é 45 minutos, 3 a 5 vezes por semana. Mais que isso, o benefício para de crescer. Muito mais que isso, ele começa a diminuir.**

Isso não é uma licença pra sedentarismo. Fazer zero exercício está associado a mais dias de sofrimento mental. Mas é uma reorganização importante do que a cabeça de quem se move considera "suficiente".

A régua que a gente costuma carregar (implícita ou explicitamente) é que "mais é sempre melhor". Quando o treino cai da semana, a gente reage como se estivesse perdendo algo proporcional à queda. Uma semana sem correr virou uma semana de humor pior. Uma pausa maior é uma dívida acumulada.

Mas a curva em U diz outra coisa. Ela diz que o ganho do exercício sobre o humor não é linear. Ele satura. E depois começa a se degradar.

Aplicando essa lente aos meus dez dias sem treino, a leitura muda. Eu não estou perdendo dez dias de "humor de corredora". Eu estou fora do que a ciência define como janela ótima, sim. Mas voltar 45 minutos, 3 vezes por semana, já me devolve a maior parte do benefício mental descrito por Chekroud. E isso é muito diferente de "preciso voltar a treinar como antes".

Há outro dado do mesmo estudo que vale trazer aqui: caminhar reduz em 17,7% os dias de sofrimento mental. Não correr. **Caminhar**. Até tarefas domésticas contam.

Ou seja: existe uma versão pequena de mim mesma se movendo hoje que já capturaria grande parte do benefício. A ideia de que "só vale se for treino sério" é folclore, não dado.

Faz sentido essa curva em U, se a gente pensar. Exercício é adaptação. Toda adaptação exige recuperação. Quando o volume é demais, o corpo não repõe. E, sem reposição, o próprio sistema que produz o benefício mental (o cortisol regulado, o BDNF, o sono restaurador) começa a falhar. O excesso vira o oposto do propósito.

*Se você olhar honestamente pra sua semana, você está no vale do U, no fundo do U, ou já subindo a rampa do outro lado?*

---

### ⚠️ Limitações do Modelo

- Estudo *cross-sectional*, mostra associação e não causa direta. Não podemos afirmar categoricamente que treinar mais de 3h por sessão *causa* piora do humor, apenas que essas duas coisas aparecem juntas na amostra.
- Saúde mental foi autorrelatada pela pergunta "quantos dias no último mês sua saúde mental não foi boa?". É uma métrica simples e útil, mas não capta transtornos específicos nem estados mentais complexos.
- Amostra é dos EUA. Cultura afeta relação com exercício, com autorrelato de saúde mental e com estilo de vida em geral. Generalizar direto pra outros contextos requer cautela.
- O achado do "45 minutos" é o ponto ótimo médio. Indivíduos podem ter janelas ótimas diferentes, especialmente atletas competitivos, pessoas com condições específicas, ou quem tem rotinas muito distintas do padrão americano.
- Este texto não substitui recomendação médica. Se você tem histórico de depressão ou ansiedade, o exercício é adjunto valioso mas não substitui tratamento clínico apropriado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
